# 배차 AI 데모 — Final v3.0 Notebook

이 노트북은 **Stateful Sequential Real-Time Dispatch Optimization** 기준의 최종 데모 파일을 한 번에 생성하고 Streamlit을 실행하기 위한 버전입니다.

- 주문 1건 접수 즉시 배차 추천
- 현재 차량 State + 기존 FIXED 일정 반영
- Hard Constraint 통과 자차별 Objective Cost 계산
- Best 자차 Cost와 용차 Fallback Cost 직접 비교
- 기본 Objective Weight: `0.60 / 0.20 / 0.10 / 0.10`
- Tuned Fallback Cost: **`1.12`**
- 15분 Buffer / Stale State 120h / Q95 이동범위 / FIXED 일정 충돌 방지
- **왼쪽 Sidebar 없음**
- 결과 화면 상단에 **`← 새 주문 입력` 버튼**
- 배차 확정 후 차량 State는 다음 주문까지 유지

> Streamlit 화면은 노트북 셀 내부가 아니라 로컬 브라우저에서 열립니다.


## 1. 패키지 설치

처음 한 번만 실행하면 됩니다. 이미 설치되어 있다면 건너뛰어도 됩니다.

In [ ]:
%pip install "streamlit>=1.37" "pandas>=2.0" "numpy>=1.24" "openpyxl>=3.1" "openai>=1.0"

## 2. 프로젝트 경로 설정

Excel 원본 파일이 있는 해커톤 폴더를 지정합니다.

이 노트북은 최종 데모를 별도 폴더인 `dispatch_ai_demo_v3_clean`에 생성하므로 이전 데모 파일과 섞이지 않습니다.

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path(r"./공모전\2026_해커톤")
DEMO_DIR = BASE_DIR / "dispatch_ai_demo_v3_clean"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

os.environ["DISPATCH_BASE_DIR"] = str(BASE_DIR)

# OpenAI API는 선택사항입니다.
# API Key가 없으면 규칙 기반 주문 parser / 후속 요청 router가 동작합니다.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
# os.environ["OPENAI_MODEL"] = "gpt-5.6-luna"

print("BASE_DIR :", BASE_DIR)
print("DEMO_DIR :", DEMO_DIR)


## 3. 입력 Excel 파일 확인

In [ ]:
required_files = [
    "모델링_최종후보표.xlsx",
    "순차모델링_FIXED스케줄.xlsx",
    "순차모델링_OD캐시.xlsx",
]

all_ok = True
for name in required_files:
    p = BASE_DIR / name
    ok = p.exists()
    all_ok &= ok
    print(f"{name}: {'OK' if ok else '없음'} -> {p}")

if not all_ok:
    print("\n위 파일이 모두 BASE_DIR에 있는지 확인하세요.")


## 4. 최종 v3.0 데모 파일 생성

아래 셀들은 `optimizer.py`, `ai_helper.py`, `app.py`, `requirements.txt`, `README.md`, `VERSION.txt`를 생성합니다.

> 기존 데모 파일을 덮어쓰지 않도록 `dispatch_ai_demo_v3_clean` 폴더를 사용합니다.

In [ ]:
%cd $DEMO_DIR
print("현재 작업 폴더:", Path.cwd())


### `optimizer.py` — Stateful Sequential 실시간 배차 엔진

In [ ]:
%%writefile optimizer.py
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from bisect import bisect_right
import copy
from typing import Any

import numpy as np
import pandas as pd


# ============================================================
# Data classes
# ============================================================
@dataclass
class Weights:
    distance: float = 0.60
    time: float = 0.20
    pickup_slack: float = 0.10
    return_slack: float = 0.10

    def normalized(self) -> "Weights":
        vals = np.array(
            [self.distance, self.time, self.pickup_slack, self.return_slack],
            dtype=float,
        )
        if np.any(vals < 0):
            raise ValueError("가중치는 0 이상이어야 합니다.")
        s = float(vals.sum())
        if s <= 0:
            raise ValueError("가중치 합은 0보다 커야 합니다.")
        vals = vals / s
        return Weights(*vals.tolist())

    def to_dict(self) -> dict[str, float]:
        return asdict(self.normalized())


@dataclass
class DemoOrder:
    pickup_rep: str
    dropoff_rep: str
    order_time: pd.Timestamp
    export_import: str | None = None
    container: str | None = None
    weight_kg: float | None = None
    line: str | None = None
    shipper: str | None = None
    order_id: str = "DEMO_ORDER"


# ============================================================
# Helpers
# ============================================================
def clean_text(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    invalid = ["", "nan", "NaN", "None", "<NA>", "null", "NULL"]
    return s.mask(s.isin(invalid))


def flag_o_value(x: Any) -> bool:
    if pd.isna(x):
        return False
    return str(x).strip().upper() == "O"


def single_or_na(series: pd.Series):
    vals = pd.unique(series.dropna())
    if len(vals) == 1:
        return vals[0]
    return pd.NA


def select_sheet(path: Path, candidates: list[str]) -> str:
    names = pd.ExcelFile(path).sheet_names
    for s in candidates:
        if s in names:
            return s
    raise KeyError(
        f"{path.name}에서 필요한 시트를 찾지 못했습니다.\n"
        f"후보={candidates}\n실제 시트={names}"
    )


def select_col(df: pd.DataFrame, candidates: list[str], required: bool = True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"필요 컬럼을 찾지 못했습니다. 후보={candidates}\n"
            f"실제 컬럼={list(df.columns)}"
        )
    return None


def slack_penalty(slack_h: float | None) -> float:
    if slack_h is None or not np.isfinite(slack_h):
        return 0.0
    return 1.0 / (1.0 + max(0.0, float(slack_h)))


# ============================================================
# Stateful Sequential Real-Time Dispatch Engine
# ============================================================
class DispatchEngine:
    """
    Final v3.0 데모 엔진.

    - 신규 주문 1건이 들어오면 즉시 처리
    - 현재 차량 State + 기존 FIXED schedule만 사용
    - Hard Constraint를 통과한 자차별 Objective Cost 계산
    - Best 자차 Cost와 Fallback Cost를 직접 비교
    - 자차 확정 시 다음 주문에서 사용할 State를 commit으로 반영

    단일 주문에서는 "운영 가능한 자차 + 용차 Fallback" 중 최소비용 선택이므로
    별도의 다중 주문 MILP가 필요하지 않습니다.
    """

    def __init__(
        self,
        base_dir: str | Path,
        model_filename: str = "모델링_최종후보표.xlsx",
        fixed_filename: str = "순차모델링_FIXED스케줄.xlsx",
        od_filename: str = "순차모델링_OD캐시.xlsx",
        buffer_h: float = 0.25,
        state_stale_cutoff_h: float = 120.0,
        min_hist_own_ratio: float = 0.50,
        default_fallback_cost: float = 1.12,
    ):
        self.base_dir = Path(base_dir)
        self.model_path = self.base_dir / model_filename
        self.fixed_path = self.base_dir / fixed_filename
        self.od_path = self.base_dir / od_filename

        self.buffer_h = float(buffer_h)
        self.state_stale_cutoff_h = float(state_stale_cutoff_h)
        self.min_hist_own_ratio = float(min_hist_own_ratio)
        self.default_fallback_cost = float(default_fallback_cost)

        self._load()

    # --------------------------------------------------------
    # Load / preprocess
    # --------------------------------------------------------
    def _load(self):
        for p in [self.model_path, self.fixed_path, self.od_path]:
            if not p.exists():
                raise FileNotFoundError(f"파일이 없습니다: {p}")

        target_sheet = select_sheet(
            self.model_path,
            ["03_순차모델링대상", "10_순차모델링대상"],
        )
        candidate_sheet = select_sheet(
            self.model_path,
            ["01_전체Feasible후보"],
        )
        full_sheet = select_sheet(
            self.fixed_path,
            ["01_전체스케줄_타겟표시"],
        )
        fixed_sheet = select_sheet(
            self.fixed_path,
            ["02_FIXED_시각스케줄", "04_FIXED_시각스케줄"],
        )
        od_sheet = select_sheet(
            self.od_path,
            ["02_ODCache"],
        )

        self.candidate_all = pd.read_excel(self.model_path, sheet_name=candidate_sheet)
        self.targets_all = pd.read_excel(self.model_path, sheet_name=target_sheet)
        self.full = pd.read_excel(self.fixed_path, sheet_name=full_sheet)
        self.fixed = pd.read_excel(self.fixed_path, sheet_name=fixed_sheet)
        self.od = pd.read_excel(self.od_path, sheet_name=od_sheet)

        for df, c in [
            (self.candidate_all, "order_time"),
            (self.targets_all, "order_time"),
            (self.targets_all, "주문_예상완료시각"),
            (self.full, "작업일시"),
            (self.fixed, "작업일시"),
        ]:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors="coerce")

        self.candidate_all["candidate_vehicle"] = clean_text(
            self.candidate_all["candidate_vehicle"]
        )
        if "candidate_hist_own_ratio" in self.candidate_all.columns:
            self.candidate_all["candidate_hist_own_ratio"] = pd.to_numeric(
                self.candidate_all["candidate_hist_own_ratio"], errors="coerce"
            )
            self.candidate_all = self.candidate_all[
                self.candidate_all["candidate_hist_own_ratio"] > self.min_hist_own_ratio
            ].copy()

        self.full["차량"] = clean_text(self.full["차량"])
        self.fixed["차량"] = clean_text(self.fixed["차량"])

        self.full_pickup_raw = select_col(
            self.full, ["현재상차지_raw", "상차지_raw"], required=False
        )
        self.full_dropoff_raw = select_col(
            self.full, ["현재하차지_raw", "하차지_raw"], required=False
        )
        self.full_pickup_rep = select_col(
            self.full, ["현재상차지_rep", "상차지_rep"]
        )
        self.full_dropoff_rep = select_col(
            self.full, ["현재하차지_rep", "하차지_rep"]
        )

        self.fixed_pickup_raw = select_col(self.fixed, ["고정상차지_raw"], required=False)
        self.fixed_dropoff_raw = select_col(self.fixed, ["고정하차지_raw"], required=False)
        self.fixed_pickup_rep = select_col(self.fixed, ["고정상차지_rep"])
        self.fixed_dropoff_rep = select_col(self.fixed, ["고정하차지_rep"])
        self.fixed_pickup_ambig = select_col(
            self.fixed, ["고정_상차지모호"], required=False
        )
        self.fixed_dropoff_ambig = select_col(
            self.fixed, ["고정_하차지모호"], required=False
        )
        self.fixed_pickup_nunique = select_col(
            self.fixed, ["고정상차지_rep_고유수"], required=False
        )
        self.fixed_dropoff_nunique = select_col(
            self.fixed, ["고정하차지_rep_고유수"], required=False
        )

        # OD cache
        for c in ["origin_rep", "destination_rep"]:
            self.od[c] = clean_text(self.od[c])
        for c in ["도로거리_km", "예상시간_h"]:
            self.od[c] = pd.to_numeric(self.od[c], errors="coerce")

        od_valid = self.od.dropna(
            subset=["origin_rep", "destination_rep", "도로거리_km", "예상시간_h"]
        ).copy()

        self.od_map = {
            (str(r.origin_rep), str(r.destination_rep)): (
                float(r.도로거리_km),
                float(r.예상시간_h),
            )
            for r in od_valid.itertuples()
        }

        # Final v3.0과 동일하게 Q75/Q95는 양수 OD에 대해 계산
        positive_dist = od_valid.loc[od_valid["도로거리_km"] > 0, "도로거리_km"]
        positive_time = od_valid.loc[od_valid["예상시간_h"] > 0, "예상시간_h"]

        self.dist_q75 = float(positive_dist.quantile(0.75))
        self.time_q75 = float(positive_time.quantile(0.75))
        self.dist_q95 = float(positive_dist.quantile(0.95))
        self.time_q95 = float(positive_time.quantile(0.95))

        if self.dist_q75 <= 0 or self.time_q75 <= 0:
            raise RuntimeError("OD Q75 scale이 0 이하입니다.")

        self.known_reps = sorted(
            set(od_valid["origin_rep"].dropna().astype(str))
            | set(od_valid["destination_rep"].dropna().astype(str))
        )

        # Candidate vehicles
        self.candidate_all = (
            self.candidate_all.dropna(subset=["order_id", "candidate_vehicle"])
            .drop_duplicates(["order_id", "candidate_vehicle"], keep="first")
            .reset_index(drop=True)
        )
        self.candidate_vehicles = sorted(
            self.candidate_all["candidate_vehicle"].dropna().astype(str).unique()
        )

        # Fixed event table
        self.fixed_by_vehicle = self._build_fixed_events()

        # 과거 자차비율 계산용 실제 배차 컬럼
        self.own_col = select_col(
            self.full,
            ["용차", "실제배차", "actual_dispatch"],
            required=False,
        )

        self.targets_all = (
            self.targets_all.dropna(subset=["order_id", "order_time"])
            .sort_values(["order_time", "order_id"], kind="mergesort")
            .drop_duplicates("order_id", keep="first")
            .reset_index(drop=True)
        )

    def _fixed_ambiguous(self, row, flag_col, n_col):
        if flag_col is not None and flag_o_value(row[flag_col]):
            return True
        if n_col is not None:
            n = pd.to_numeric(row[n_col], errors="coerce")
            if pd.notna(n) and float(n) > 1:
                return True
        return False

    def _build_fixed_events(self):
        out = {}
        fixed_sorted = self.fixed.sort_values(["차량", "작업일시"])
        for vehicle, g in fixed_sorted.groupby("차량", sort=False):
            events = []
            for _, r in g.iterrows():
                events.append(
                    {
                        "time": r["작업일시"],
                        "pickup_raw": r[self.fixed_pickup_raw]
                        if self.fixed_pickup_raw else pd.NA,
                        "pickup_rep": r[self.fixed_pickup_rep],
                        "dropoff_raw": r[self.fixed_dropoff_raw]
                        if self.fixed_dropoff_raw else pd.NA,
                        "dropoff_rep": r[self.fixed_dropoff_rep],
                        "pickup_ambiguous": self._fixed_ambiguous(
                            r, self.fixed_pickup_ambig, self.fixed_pickup_nunique
                        ),
                        "dropoff_ambiguous": self._fixed_ambiguous(
                            r, self.fixed_dropoff_ambig, self.fixed_dropoff_nunique
                        ),
                    }
                )
            out[str(vehicle)] = events
        return out

    # --------------------------------------------------------
    # Public info
    # --------------------------------------------------------
    def data_summary(self) -> dict[str, Any]:
        return {
            "target_n": len(self.targets_all),
            "candidate_pair_n": len(self.candidate_all),
            "candidate_vehicle_n": len(self.candidate_vehicles),
            "od_pair_n": len(self.od_map),
            "dist_q75": self.dist_q75,
            "time_q75": self.time_q75,
            "dist_q95": self.dist_q95,
            "time_q95": self.time_q95,
            "fallback_cost": self.default_fallback_cost,
        }

    def get_od(self, origin_rep, destination_rep):
        if pd.isna(origin_rep) or pd.isna(destination_rep):
            return np.nan, np.nan
        origin_rep = str(origin_rep)
        destination_rep = str(destination_rep)
        if origin_rep == destination_rep:
            return 0.0, 0.0
        return self.od_map.get((origin_rep, destination_rep), (np.nan, np.nan))

    def nearest_rep(self, text: str) -> str | None:
        if not text:
            return None
        t = str(text).strip()
        if t in self.known_reps:
            return t
        hits = [r for r in self.known_reps if t in r or r in t]
        return hits[0] if hits else None

    # --------------------------------------------------------
    # States
    # --------------------------------------------------------
    def _latest_actual_state(self, vehicle: str, before_time: pd.Timestamp):
        g = self.full[
            (self.full["차량"].astype(str) == str(vehicle))
            & (self.full["작업일시"] < before_time)
        ].copy()

        events = self.fixed_by_vehicle.get(str(vehicle), [])
        fixed_times = [e["time"] for e in events]

        if len(g) == 0:
            return {
                "available_time": pd.NaT,
                "anchor_time": pd.NaT,
                "location_raw": pd.NA,
                "location_rep": pd.NA,
                "source": "unknown",
                "last_order_id": pd.NA,
                "fixed_idx": 0,
            }

        last_time = g["작업일시"].max()
        same = g[g["작업일시"] == last_time]
        rep = single_or_na(same[self.full_dropoff_rep])
        raw = (
            single_or_na(same[self.full_dropoff_raw])
            if self.full_dropoff_raw
            else pd.NA
        )

        return {
            "available_time": last_time,
            "anchor_time": last_time,
            "location_raw": raw,
            "location_rep": rep,
            "source": "initial_actual",
            "last_order_id": pd.NA,
            "fixed_idx": bisect_right(fixed_times, last_time),
        }

    def _apply_commits(
        self,
        state: dict,
        vehicle: str,
        current_time: pd.Timestamp,
        commits: list[dict] | None,
    ):
        if not commits:
            return state

        # 이미 이전에 확정한 주문이면, 아직 수행 중이어도 available_time=finish_time으로 반영한다.
        relevant = []
        for c in commits:
            if c.get("type") != "own":
                continue
            if str(c.get("vehicle")) != str(vehicle):
                continue
            order_time = pd.to_datetime(c.get("order_time"), errors="coerce")
            if pd.notna(order_time) and order_time < current_time:
                relevant.append(c)

        if not relevant:
            return state

        relevant = sorted(
            relevant,
            key=lambda c: pd.to_datetime(c.get("order_time"), errors="coerce"),
        )

        for c in relevant:
            finish = pd.to_datetime(c.get("finish_time"), errors="coerce")
            if pd.isna(finish):
                continue

            # demo에서 확정한 State가 더 최신/미래 가용상태이면 우선 반영
            if pd.isna(state["available_time"]) or finish > state["available_time"]:
                events = self.fixed_by_vehicle.get(str(vehicle), [])
                fixed_times = [e["time"] for e in events]
                state = {
                    "available_time": finish,
                    "anchor_time": finish,
                    "location_raw": c.get("dropoff_raw", pd.NA),
                    "location_rep": c.get("dropoff_rep", pd.NA),
                    "source": "simulated_target",
                    "last_order_id": c.get("order_id", pd.NA),
                    "fixed_idx": bisect_right(fixed_times, finish),
                }

        return state

    def build_states(self, current_time: pd.Timestamp, commits=None):
        states = {}
        for vehicle in self.candidate_vehicles:
            s = self._latest_actual_state(vehicle, current_time)
            s = self._apply_commits(s, vehicle, current_time, commits)
            states[str(vehicle)] = s
        return states

    def project_state(self, vehicle, current_time, states):
        vehicle = str(vehicle)
        s = copy.deepcopy(states[vehicle])
        events = self.fixed_by_vehicle.get(vehicle, [])
        idx = int(s.get("fixed_idx", 0))

        # 현재 주문 이전에 발생한 FIXED 작업은 state에 반영
        while idx < len(events) and events[idx]["time"] < current_time:
            ev = events[idx]
            s["available_time"] = ev["time"]
            s["anchor_time"] = ev["time"]
            if ev["dropoff_ambiguous"]:
                s["location_raw"] = pd.NA
                s["location_rep"] = pd.NA
            else:
                s["location_raw"] = ev["dropoff_raw"]
                s["location_rep"] = ev["dropoff_rep"]
            s["source"] = "fixed"
            s["last_order_id"] = pd.NA
            idx += 1

        s["fixed_idx"] = idx
        return s

    def next_fixed(self, vehicle, projected_state):
        events = self.fixed_by_vehicle.get(str(vehicle), [])
        idx = int(projected_state.get("fixed_idx", 0))
        if idx >= len(events):
            return None
        return events[idx]

    def state_age_h(self, state, current_time):
        if state["source"] in {"initial_actual", "fixed"} and pd.notna(state["anchor_time"]):
            return max(
                0.0,
                (current_time - state["anchor_time"]).total_seconds() / 3600,
            )
        return 0.0

    # --------------------------------------------------------
    # Candidate pool for a new order
    # --------------------------------------------------------
    def custom_candidate_vehicles(self, order_time: pd.Timestamp) -> list[str]:
        """주문시각 이전 실제 이력에서 자차 비율 > 50%인 후보 차량만 사용."""
        hist = self.full[self.full["작업일시"] < order_time].copy()
        if self.own_col is None:
            return self.candidate_vehicles.copy()

        hist["_is_own"] = hist[self.own_col].astype(str).str.strip().eq("자차")
        ratio = hist.groupby("차량")["_is_own"].mean()
        valid = set(ratio[ratio > self.min_hist_own_ratio].index.astype(str))
        return [v for v in self.candidate_vehicles if str(v) in valid]

    # --------------------------------------------------------
    # Pair calculation
    # --------------------------------------------------------
    def make_pair_info(
        self,
        order: dict | pd.Series,
        vehicle: str,
        projected_state: dict,
        weights: Weights,
    ):
        weights = weights.normalized()
        t = pd.to_datetime(order["order_time"])
        pickup_rep = order["order_pickup_rep"]
        dropoff_rep = order["order_dropoff_rep"]
        finish_time = pd.to_datetime(order["주문_예상완료시각"])

        if pd.isna(projected_state["available_time"]) or pd.isna(projected_state["location_rep"]):
            return None, "state_unknown"

        age_h = self.state_age_h(projected_state, t)
        if (
            projected_state["source"] in {"initial_actual", "fixed"}
            and age_h > self.state_stale_cutoff_h + 1e-9
        ):
            return None, "stale_state"

        empty_km, empty_h = self.get_od(projected_state["location_rep"], pickup_rep)
        if pd.isna(empty_km) or pd.isna(empty_h):
            return None, "pickup_od_missing"

        if empty_km > self.dist_q95 + 1e-9 or empty_h > self.time_q95 + 1e-9:
            return None, "q95_range"

        arrival_time = projected_state["available_time"] + pd.Timedelta(hours=float(empty_h))
        pickup_ready = arrival_time + pd.Timedelta(hours=self.buffer_h)
        pickup_slack_h = (t - pickup_ready).total_seconds() / 3600
        if pickup_slack_h < -1e-9:
            return None, "pickup_time_fail"

        next_fixed = self.next_fixed(vehicle, projected_state)
        next_fixed_time = pd.NaT
        return_km = np.nan
        return_h = np.nan
        return_slack_h = np.inf

        if next_fixed is not None:
            next_fixed_time = next_fixed["time"]
            if next_fixed["pickup_ambiguous"] or pd.isna(next_fixed["pickup_rep"]):
                return None, "fixed_ambiguous"

            return_km, return_h = self.get_od(dropoff_rep, next_fixed["pickup_rep"])
            if pd.isna(return_km) or pd.isna(return_h):
                return None, "fixed_return_od_missing"

            return_ready = finish_time + pd.Timedelta(hours=float(return_h) + self.buffer_h)
            return_slack_h = (next_fixed_time - return_ready).total_seconds() / 3600
            if return_slack_h < -1e-9:
                return None, "fixed_conflict"

        d_norm = float(empty_km) / self.dist_q75
        t_norm = float(empty_h) / self.time_q75
        p_pen = slack_penalty(pickup_slack_h)
        r_pen = slack_penalty(return_slack_h)

        objective_cost = (
            weights.distance * d_norm
            + weights.time * t_norm
            + weights.pickup_slack * p_pen
            + weights.return_slack * r_pen
        )

        return {
            "vehicle": str(vehicle),
            "state_source": projected_state["source"],
            "state_anchor_time": projected_state["anchor_time"],
            "state_age_h": float(age_h),
            "before_available_time": projected_state["available_time"],
            "before_raw": projected_state["location_raw"],
            "before_rep": projected_state["location_rep"],
            "empty_km": float(empty_km),
            "empty_h": float(empty_h),
            "arrival_time": arrival_time,
            "pickup_slack_h": float(pickup_slack_h),
            "next_fixed_time": next_fixed_time,
            "return_km": return_km,
            "return_h": return_h,
            "return_slack_h": return_slack_h,
            "distance_norm": d_norm,
            "time_norm": t_norm,
            "pickup_penalty": p_pen,
            "return_penalty": r_pen,
            "objective_cost": float(objective_cost),
        }, None

    # --------------------------------------------------------
    # Build demo order
    # --------------------------------------------------------
    def build_demo_order_row(self, order: DemoOrder) -> dict:
        pickup_rep = self.nearest_rep(order.pickup_rep)
        dropoff_rep = self.nearest_rep(order.dropoff_rep)
        if pickup_rep is None:
            raise ValueError(f"상차 대표위치를 찾지 못했습니다: {order.pickup_rep}")
        if dropoff_rep is None:
            raise ValueError(f"하차 대표위치를 찾지 못했습니다: {order.dropoff_rep}")

        loaded_km, loaded_h = self.get_od(pickup_rep, dropoff_rep)
        if pd.isna(loaded_h):
            raise ValueError(f"적재 구간 OD가 없습니다: {pickup_rep} → {dropoff_rep}")

        t = pd.to_datetime(order.order_time)
        finish = t + pd.Timedelta(hours=float(loaded_h))
        return {
            "order_id": order.order_id,
            "order_time": t,
            "order_pickup_rep": pickup_rep,
            "order_dropoff_rep": dropoff_rep,
            "order_pickup_raw": pickup_rep,
            "order_dropoff_raw": dropoff_rep,
            "주문_적재거리_km": float(loaded_km),
            "주문_적재시간_h": float(loaded_h),
            "주문_예상완료시각": finish,
            "수출입": order.export_import,
            "컨테이너": order.container,
            "중량": order.weight_kg,
            "라인": order.line,
            "화주": order.shipper,
        }

    # --------------------------------------------------------
    # One live recommendation: Final v3.0 Sequential logic
    # --------------------------------------------------------
    def recommend(
        self,
        order: DemoOrder,
        weights: Weights | None = None,
        fallback_cost: float | None = None,
        commits: list[dict] | None = None,
        top_n: int = 5,
    ) -> dict[str, Any]:
        weights = (weights or Weights()).normalized()
        fallback_cost = float(
            self.default_fallback_cost if fallback_cost is None else fallback_cost
        )

        current = self.build_demo_order_row(order)
        t0 = current["order_time"]
        states = self.build_states(t0, commits=commits)
        pool = self.custom_candidate_vehicles(t0)

        feasible = []
        exclude_reasons: dict[str, int] = {}

        for vehicle in pool:
            if vehicle not in states:
                exclude_reasons["state_unknown"] = exclude_reasons.get("state_unknown", 0) + 1
                continue

            projected = self.project_state(vehicle, t0, states)
            info, reason = self.make_pair_info(current, vehicle, projected, weights)
            if info is None:
                exclude_reasons[reason] = exclude_reasons.get(reason, 0) + 1
                continue
            feasible.append(info)

        # Final v3.0과 동일한 deterministic sorting
        feasible = sorted(
            feasible,
            key=lambda x: (
                x["objective_cost"],
                x["empty_km"],
                x["empty_h"],
                -x["pickup_slack_h"],
                -(x["return_slack_h"] if np.isfinite(x["return_slack_h"]) else 1e12),
                x["vehicle"],
            ),
        )
        for i, row in enumerate(feasible, start=1):
            row["rank"] = i

        best = feasible[0] if feasible else None

        if best is None:
            recommend_type = "용차"
            selected_vehicle = None
            selected_info = None
            reason = "Hard Constraint를 만족하는 운영 가능한 자차가 없어 용차 Fallback을 추천했습니다."
        elif best["objective_cost"] <= fallback_cost + 1e-9:
            recommend_type = "자차"
            selected_vehicle = best["vehicle"]
            selected_info = best
            reason = (
                f"{selected_vehicle}가 모든 Hard Constraint를 만족한 자차 중 "
                f"Objective Cost가 가장 낮고({best['objective_cost']:.3f}), "
                f"Fallback Cost {fallback_cost:.2f} 이하이므로 추천되었습니다."
            )
        else:
            recommend_type = "용차"
            selected_vehicle = None
            selected_info = None
            reason = (
                f"운영 가능한 자차는 있지만 가장 낮은 자차 Objective Cost가 "
                f"{best['objective_cost']:.3f}로 Fallback Cost {fallback_cost:.2f}보다 높아 "
                f"용차를 추천했습니다."
            )

        return {
            "order": current,
            "weights": weights.to_dict(),
            "fallback_cost": fallback_cost,
            "recommend_type": recommend_type,
            "recommended_vehicle": selected_vehicle,
            "selected_info": selected_info,
            "best_own_info": best,
            "best_own_cost": best["objective_cost"] if best is not None else np.nan,
            "top_candidates": feasible[:top_n],
            "all_feasible_candidates": feasible,
            "historical_candidate_n": len(pool),
            "feasible_candidate_n": len(feasible),
            "fallback_selected": recommend_type == "용차",
            "exclude_reasons": exclude_reasons,
            "reason": reason,
        }

    # --------------------------------------------------------
    # Session commit
    # --------------------------------------------------------
    def build_commit(self, result: dict) -> dict:
        order = result["order"]
        if result.get("recommend_type") != "자차":
            return {
                "order_id": order["order_id"],
                "type": "fallback",
                "order_time": order["order_time"],
            }

        info = result["selected_info"]
        return {
            "order_id": order["order_id"],
            "type": "own",
            "vehicle": result["recommended_vehicle"],
            "order_time": order["order_time"],
            "finish_time": order["주문_예상완료시각"],
            "dropoff_rep": order["order_dropoff_rep"],
            "dropoff_raw": order.get("order_dropoff_raw", order["order_dropoff_rep"]),
            "objective_cost": info["objective_cost"],
        }


### `ai_helper.py` — 자연어 주문 파싱 / 후속 대화

In [ ]:
%%writefile ai_helper.py
from __future__ import annotations

import json
import os
import re
from typing import Any

import pandas as pd

try:
    from openai import OpenAI
except Exception:
    OpenAI = None


DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")


def _client():
    if OpenAI is None:
        return None
    if not os.getenv("OPENAI_API_KEY"):
        return None
    return OpenAI()


def _strip_json_fence(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


def parse_order_text(text: str, known_reps: list[str]) -> dict[str, Any]:
    """자연어 주문을 구조화. API key가 없으면 규칙 기반 parser 사용."""
    client = _client()
    if client is None:
        return fallback_parse_order(text, known_reps)

    reps_preview = known_reps[:300]
    instructions = f"""
너는 컨테이너 물류 배차 시스템의 주문 파서다.
사용자 문장에서 주문 정보를 추출하고 반드시 JSON 객체 하나만 반환한다.
설명문, markdown, 코드블록은 금지한다.

사용자가 날짜를 명확히 말하지 않았다면 order_time은 null로 둔다.
상차/하차 위치는 가능한 경우 아래 representative location 목록 중 정확히 하나를 골라라.
목록에 명확한 대응이 없으면 사용자가 쓴 원문을 그대로 넣어라.

대표 위치 목록 일부:
{reps_preview}

반환 스키마:
{{
  "pickup_rep": string|null,
  "dropoff_rep": string|null,
  "order_time": "YYYY-MM-DD HH:MM"|null,
  "export_import": string|null,
  "container": string|null,
  "weight_kg": number|null,
  "line": string|null,
  "shipper": string|null
}}

중량이 톤이면 kg으로 변환한다. 예: 18톤 -> 18000.
""".strip()

    response = client.responses.create(
        model=DEFAULT_MODEL,
        reasoning={"effort": "low"},
        instructions=instructions,
        input=text,
    )
    raw = _strip_json_fence(response.output_text)
    try:
        return json.loads(raw)
    except Exception as e:
        raise ValueError(f"주문 JSON 파싱 실패: {raw}") from e


def classify_followup(text: str) -> dict[str, Any]:
    """후속 요청을 explain / compare / reoptimize / confirm / new_order로 분류."""
    client = _client()
    if client is None:
        return fallback_classify_followup(text)

    instructions = """
너는 물류 배차 데모의 대화 라우터다.
사용자의 후속 문장을 아래 intent 중 하나로 분류하고 JSON 하나만 반환한다.
- explain: 추천 이유/근거/왜 질문
- compare: 후보 차량 비교 요청
- reoptimize: 거리/시간/다음 일정/여유 등 선호 변경 후 다시 추천 요청
- confirm: 현재 추천 확정 요청
- new_order: 새 주문을 입력하고 싶다는 요청

reoptimize이면 preference는 아래 중 하나:
- balanced
- distance_first
- time_first
- schedule_safe
- custom

사용자가 명시적으로 가중치를 말하면 custom_weights에
{"distance":..., "time":..., "pickup_slack":..., "return_slack":...} 형태로 넣는다.
합이 1이 아니어도 된다.

compare이면 candidate_rank에 비교 대상 순위(예: 2)를 넣는다. 없으면 null.

반환 JSON:
{
  "intent": "explain|compare|reoptimize|confirm|new_order",
  "preference": null|string,
  "custom_weights": null|object,
  "candidate_rank": null|integer
}
""".strip()

    response = client.responses.create(
        model=DEFAULT_MODEL,
        reasoning={"effort": "low"},
        instructions=instructions,
        input=text,
    )
    raw = _strip_json_fence(response.output_text)
    try:
        return json.loads(raw)
    except Exception as e:
        raise ValueError(f"Intent JSON 파싱 실패: {raw}") from e


def explain_result(user_text: str, result: dict[str, Any]) -> str:
    """현재 최적화 결과의 숫자만 근거로 설명."""
    client = _client()
    if client is None:
        return deterministic_explanation(result)

    selected = result.get("selected_info") or {}
    best = result.get("best_own_info") or {}
    top = result.get("top_candidates") or []
    safe_context = {
        "recommend_type": result.get("recommend_type"),
        "recommended_vehicle": result.get("recommended_vehicle"),
        "fallback_cost": result.get("fallback_cost"),
        "weights": result.get("weights"),
        "reason": result.get("reason"),
        "selected_info": {
            "before_rep": selected.get("before_rep"),
            "empty_km": selected.get("empty_km"),
            "empty_h": selected.get("empty_h"),
            "pickup_slack_h": selected.get("pickup_slack_h"),
            "next_fixed_time": str(selected.get("next_fixed_time")),
            "return_slack_h": selected.get("return_slack_h"),
            "objective_cost": selected.get("objective_cost"),
        },
        "best_own_info": {
            "vehicle": best.get("vehicle"),
            "objective_cost": best.get("objective_cost"),
            "empty_km": best.get("empty_km"),
            "empty_h": best.get("empty_h"),
        },
        "top_candidates": [
            {
                "rank": c.get("rank"),
                "vehicle": c.get("vehicle"),
                "before_rep": c.get("before_rep"),
                "empty_km": c.get("empty_km"),
                "empty_h": c.get("empty_h"),
                "objective_cost": c.get("objective_cost"),
                "pickup_slack_h": c.get("pickup_slack_h"),
                "return_slack_h": c.get("return_slack_h"),
            }
            for c in top
        ],
    }

    instructions = """
너는 물류 운영자에게 실시간 순차 배차 최적화 결과를 설명하는 AI다.
반드시 제공된 JSON 데이터만 근거로 답한다. 없는 숫자나 사실은 만들지 않는다.
최종 추천은 생성형 AI가 임의로 고른 것이 아니라,
Hard Constraint를 통과한 자차의 Objective Cost와 용차 Fallback Cost를 비교한 최적화 엔진의 결과다.
운영자가 이해하기 쉬운 한국어로 짧고 명확하게 답한다.
""".strip()

    response = client.responses.create(
        model=DEFAULT_MODEL,
        reasoning={"effort": "low"},
        instructions=instructions,
        input=(
            f"사용자 질문: {user_text}\n\n"
            f"최적화 결과 JSON:\n{json.dumps(safe_context, ensure_ascii=False, default=str)}"
        ),
    )
    return response.output_text.strip()


def fallback_parse_order(text: str, known_reps: list[str]) -> dict[str, Any]:
    reps = sorted(known_reps, key=len, reverse=True)
    found = [r for r in reps if r in text]
    pickup = found[0] if len(found) >= 1 else None
    dropoff = found[1] if len(found) >= 2 else None

    weight_kg = None
    m = re.search(r"(\d+(?:\.\d+)?)\s*톤", text)
    if m:
        weight_kg = float(m.group(1)) * 1000
    else:
        m = re.search(r"(\d+(?:\.\d+)?)\s*kg", text, re.I)
        if m:
            weight_kg = float(m.group(1))

    container = None
    m = re.search(r"\b(20|40)\s*(HC|GP|HQ|FT)?\b", text, re.I)
    if m:
        container = (m.group(1) + (m.group(2) or "")).upper()

    export_import = "수출" if "수출" in text else ("수입" if "수입" in text else None)

    order_time = None
    m = re.search(r"(20\d{2})[-./](\d{1,2})[-./](\d{1,2})\s+(\d{1,2}):(\d{2})", text)
    if m:
        order_time = (
            f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d} "
            f"{int(m.group(4)):02d}:{int(m.group(5)):02d}"
        )

    return {
        "pickup_rep": pickup,
        "dropoff_rep": dropoff,
        "order_time": order_time,
        "export_import": export_import,
        "container": container,
        "weight_kg": weight_kg,
        "line": None,
        "shipper": None,
    }


def fallback_classify_followup(text: str) -> dict[str, Any]:
    t = text.strip()
    if any(k in t for k in ["새 주문", "다른 주문", "처음으로", "처음 화면"]):
        return {"intent": "new_order", "preference": None, "custom_weights": None, "candidate_rank": None}
    if any(k in t for k in ["확정", "배차해", "이 차량으로", "그 차량으로"]):
        return {"intent": "confirm", "preference": None, "custom_weights": None, "candidate_rank": None}
    if "비교" in t:
        m = re.search(r"(\d+)\s*번", t)
        return {"intent": "compare", "preference": None, "custom_weights": None, "candidate_rank": int(m.group(1)) if m else 2}
    if any(k in t for k in ["다음 일정", "복귀", "일정 여유"]):
        return {"intent": "reoptimize", "preference": "schedule_safe", "custom_weights": None, "candidate_rank": None}
    if any(k in t for k in ["시간", "빨리", "급해"]):
        return {"intent": "reoptimize", "preference": "time_first", "custom_weights": None, "candidate_rank": None}
    if any(k in t for k in ["거리", "공차"]):
        return {"intent": "reoptimize", "preference": "distance_first", "custom_weights": None, "candidate_rank": None}
    if any(k in t for k in ["왜", "근거", "이유", "설명"]):
        return {"intent": "explain", "preference": None, "custom_weights": None, "candidate_rank": None}
    return {"intent": "explain", "preference": None, "custom_weights": None, "candidate_rank": None}


def deterministic_explanation(result: dict[str, Any]) -> str:
    if result.get("recommend_type") != "자차":
        return result.get("reason", "용차 Fallback이 선택되었습니다.")

    info = result.get("selected_info") or {}
    v = result.get("recommended_vehicle")
    empty_km = info.get("empty_km")
    empty_h = info.get("empty_h")
    p = info.get("pickup_slack_h")
    r = info.get("return_slack_h")
    cost = info.get("objective_cost")
    fallback = result.get("fallback_cost")

    parts = [f"{v}가 가장 적합한 자차로 선택되었습니다."]
    if cost is not None:
        parts.append(f"Objective Cost는 {cost:.3f}이고 Fallback Cost {fallback:.2f} 이하입니다.")
    if empty_km is not None:
        parts.append(f"상차지까지 공차거리는 약 {empty_km:.1f}km입니다.")
    if empty_h is not None:
        parts.append(f"예상 공차시간은 약 {empty_h * 60:.0f}분입니다.")
    if p is not None:
        parts.append(f"15분 Buffer 적용 후 상차 여유는 약 {p:.2f}시간입니다.")
    if r is not None and pd.notna(r) and r != float("inf"):
        parts.append(f"다음 FIXED 일정 복귀 여유는 약 {r:.2f}시간입니다.")
    return " ".join(parts)


def preset_weights(name: str) -> dict[str, float]:
    presets = {
        "balanced": {"distance": 0.60, "time": 0.20, "pickup_slack": 0.10, "return_slack": 0.10},
        "distance_first": {"distance": 0.70, "time": 0.15, "pickup_slack": 0.075, "return_slack": 0.075},
        "time_first": {"distance": 0.40, "time": 0.40, "pickup_slack": 0.10, "return_slack": 0.10},
        "schedule_safe": {"distance": 0.40, "time": 0.15, "pickup_slack": 0.15, "return_slack": 0.30},
    }
    return presets.get(name, presets["balanced"])


### `app.py` — Sidebar 없는 Streamlit UI + 새 주문 버튼

In [ ]:
%%writefile app.py
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

from optimizer import DispatchEngine, DemoOrder, Weights
from ai_helper import (
    parse_order_text,
    classify_followup,
    explain_result,
    preset_weights,
)


# ============================================================
# Page config
# ============================================================
st.set_page_config(
    page_title="배차 AI",
    page_icon="🚚",
    layout="wide",
    initial_sidebar_state="collapsed",
)

st.markdown(
    """
<style>
[data-testid="stSidebar"] { display: none; }
[data-testid="collapsedControl"] { display: none; }
.block-container {
    padding-top: 1.6rem;
    padding-bottom: 2.5rem;
    max-width: 1180px;
}
.kpi-card {
    border: 1px solid #e5e7eb;
    border-radius: 16px;
    padding: 18px 20px;
    background: white;
    min-height: 118px;
    box-shadow: 0 1px 3px rgba(15, 23, 42, 0.04);
}
.kpi-title { color:#64748b; font-size:13px; font-weight:700; }
.kpi-value { color:#172033; font-size:29px; font-weight:800; margin-top:8px; }
.kpi-sub { color:#7c8799; font-size:12px; margin-top:7px; }
.reason-box {
    border-left:4px solid #2563eb;
    background:#f7faff;
    padding:15px 17px;
    border-radius:9px;
    line-height:1.75;
}
.small-note { color:#6b7280; font-size:12px; }
[data-testid="stChatInput"] { border-radius: 14px; }
</style>
""",
    unsafe_allow_html=True,
)


# ============================================================
# Configuration
# ============================================================
DEFAULT_BASE_DIR = r"./공모전\2026_해커톤"
BASE_DIR = Path(os.getenv("DISPATCH_BASE_DIR", DEFAULT_BASE_DIR))
DEFAULT_FALLBACK_COST = 1.12


@st.cache_resource(show_spinner="배차 데이터를 불러오는 중입니다...")
def load_engine(base_dir: str):
    return DispatchEngine(
        base_dir=base_dir,
        default_fallback_cost=DEFAULT_FALLBACK_COST,
    )


# ============================================================
# Session state
# ============================================================
def ss_init():
    defaults = {
        "messages": [],
        "parsed_order": None,
        "current_order": None,
        "current_result": None,
        "weights": preset_weights("balanced"),
        "fallback_cost": DEFAULT_FALLBACK_COST,
        "commits": [],
        "confirmed": False,
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v


ss_init()


def reset_for_new_order(keep_commits: bool = True):
    """다음 주문 입력 화면으로 이동. 확정 배차 State는 기본적으로 유지."""
    st.session_state.messages = []
    st.session_state.parsed_order = None
    st.session_state.current_order = None
    st.session_state.current_result = None
    st.session_state.weights = preset_weights("balanced")
    st.session_state.confirmed = False
    if not keep_commits:
        st.session_state.commits = []


def weights_obj():
    w = st.session_state.weights
    return Weights(
        distance=float(w["distance"]),
        time=float(w["time"]),
        pickup_slack=float(w["pickup_slack"]),
        return_slack=float(w["return_slack"]),
    ).normalized()


# ============================================================
# Header first, then engine load
# ============================================================
header_left, header_right = st.columns([0.78, 0.22])
with header_left:
    st.title("🚚 배차 AI")
    st.caption("자연어 주문 입력 → Stateful Sequential 실시간 최적화 → 결과 설명/재추천")

with header_right:
    if st.session_state.current_result is not None:
        if st.button("← 새 주문 입력", use_container_width=True):
            reset_for_new_order(keep_commits=True)
            st.rerun()

try:
    engine = load_engine(str(BASE_DIR))
except Exception as e:
    st.error("배차 엔진을 불러오지 못했습니다.")
    st.code(str(e))
    st.info(
        "app.py의 DEFAULT_BASE_DIR 또는 환경변수 DISPATCH_BASE_DIR를 "
        "Excel 파일이 있는 폴더로 맞춰주세요."
    )
    st.stop()

info = engine.data_summary()


def rerun_current():
    order = st.session_state.current_order
    if order is None:
        return
    with st.spinner("현재 차량 State를 기준으로 최적 배차를 계산하고 있습니다..."):
        st.session_state.current_result = engine.recommend(
            order=order,
            weights=weights_obj(),
            fallback_cost=st.session_state.fallback_cost,
            commits=st.session_state.commits,
            top_n=5,
        )
    st.session_state.confirmed = False


# ============================================================
# Initial order input
# ============================================================
if st.session_state.current_result is None:
    st.write("")
    st.subheader("신규 오더 입력")
    st.write(
        "예: **2026-03-18 09:00 부산신항에서 의왕ICD로 40HC 18톤 수출 건 배차해줘**"
    )
    st.caption(
        f"Historical Replay 데모 · 후보 차량 {info['candidate_vehicle_n']:,}대 · "
        f"Fallback Cost {st.session_state.fallback_cost:.2f}"
    )

    user_text = st.chat_input("배차할 주문을 자연어로 입력하세요")

    if user_text:
        st.session_state.messages.append({"role": "user", "content": user_text})
        try:
            st.session_state.parsed_order = parse_order_text(user_text, engine.known_reps)
        except Exception as e:
            st.error(f"주문 해석 실패: {e}")

    parsed = st.session_state.parsed_order
    if parsed is not None:
        st.markdown("### AI가 이해한 주문")

        c1, c2, c3 = st.columns(3)
        with c1:
            pickup_guess = engine.nearest_rep(parsed.get("pickup_rep"))
            dropoff_guess = engine.nearest_rep(parsed.get("dropoff_rep"))

            pickup = st.selectbox(
                "상차지",
                options=engine.known_reps,
                index=engine.known_reps.index(pickup_guess)
                if pickup_guess in engine.known_reps else 0,
            )
            dropoff = st.selectbox(
                "하차지",
                options=engine.known_reps,
                index=engine.known_reps.index(dropoff_guess)
                if dropoff_guess in engine.known_reps else 0,
            )

        with c2:
            parsed_dt = pd.to_datetime(parsed.get("order_time"), errors="coerce")
            if pd.isna(parsed_dt):
                parsed_dt = pd.to_datetime(engine.targets_all["order_time"].median())
            order_date = st.date_input("작업일", value=parsed_dt.date())
            order_time = st.time_input(
                "작업시간",
                value=parsed_dt.time().replace(second=0, microsecond=0),
            )

        with c3:
            export_import = st.selectbox(
                "수출입",
                ["수출", "수입"],
                index=0 if parsed.get("export_import") != "수입" else 1,
            )
            container = st.text_input(
                "컨테이너",
                value=str(parsed.get("container") or "40HC"),
            )
            weight_kg = st.number_input(
                "중량(kg)",
                min_value=0.0,
                value=float(parsed.get("weight_kg") or 18000.0),
                step=100.0,
            )

        if st.button("AI 배차 시작 →", type="primary", use_container_width=True):
            dt = pd.Timestamp.combine(order_date, order_time)
            demo_order = DemoOrder(
                pickup_rep=pickup,
                dropoff_rep=dropoff,
                order_time=dt,
                export_import=export_import,
                container=container,
                weight_kg=weight_kg,
                line=parsed.get("line"),
                shipper=parsed.get("shipper"),
                order_id=f"DEMO_{dt.strftime('%Y%m%d_%H%M%S')}_{len(st.session_state.commits)+1:02d}",
            )
            st.session_state.current_order = demo_order
            rerun_current()
            st.rerun()

    st.stop()


# ============================================================
# Result dashboard
# ============================================================
result = st.session_state.current_result
order = result["order"]
selected = result.get("selected_info") or {}
best_own = result.get("best_own_info") or {}

st.write("")
st.subheader("AI 배차 결과")
st.caption(
    f"{pd.to_datetime(order['order_time']).strftime('%Y-%m-%d %H:%M')} · "
    f"{order['order_pickup_rep']} → {order['order_dropoff_rep']} · "
    "주문 접수 즉시 현재 차량 State 기준으로 계산"
)

k1, k2, k3, k4 = st.columns(4)

with k1:
    if result["recommend_type"] == "자차":
        sub = result.get("recommended_vehicle") or "-"
    else:
        sub = f"Fallback Cost {result['fallback_cost']:.2f}"
    st.markdown(
        f'<div class="kpi-card"><div class="kpi-title">추천 유형</div>'
        f'<div class="kpi-value">{result["recommend_type"]}</div>'
        f'<div class="kpi-sub">{sub}</div></div>',
        unsafe_allow_html=True,
    )

with k2:
    km = selected.get("empty_km")
    km_txt = f"{km:.1f} km" if km is not None else "-"
    st.markdown(
        f'<div class="kpi-card"><div class="kpi-title">예상 공차거리</div>'
        f'<div class="kpi-value">{km_txt}</div>'
        f'<div class="kpi-sub">Q95 제한 {engine.dist_q95:.1f} km</div></div>',
        unsafe_allow_html=True,
    )

with k3:
    h = selected.get("empty_h")
    min_txt = f"{h * 60:.0f}분" if h is not None else "-"
    st.markdown(
        f'<div class="kpi-card"><div class="kpi-title">예상 공차시간</div>'
        f'<div class="kpi-value">{min_txt}</div>'
        f'<div class="kpi-sub">15분 Buffer 적용</div></div>',
        unsafe_allow_html=True,
    )

with k4:
    st.markdown(
        f'<div class="kpi-card"><div class="kpi-title">가용 자차 후보</div>'
        f'<div class="kpi-value">{result["feasible_candidate_n"]}대</div>'
        f'<div class="kpi-sub">Best 자차 vs Fallback {result["fallback_cost"]:.2f}</div></div>',
        unsafe_allow_html=True,
    )

st.write("")
left, right = st.columns([0.68, 0.32])

with left:
    st.markdown("### 추천 차량 순위")
    cand = pd.DataFrame(result["top_candidates"])
    if len(cand) == 0:
        st.warning("Hard Constraint를 만족하는 자차 후보가 없습니다.")
    else:
        next_slack = cand["return_slack_h"].replace(np.inf, np.nan)
        show = pd.DataFrame(
            {
                "순위": cand["rank"],
                "차량번호": cand["vehicle"],
                "현재 위치": cand["before_rep"],
                "공차거리(km)": cand["empty_km"].round(1),
                "공차시간(분)": (cand["empty_h"] * 60).round(0).astype(int),
                "Objective Cost": cand["objective_cost"].round(3),
                "상차여유(h)": cand["pickup_slack_h"].round(2),
                "다음일정여유(h)": next_slack.round(2),
            }
        )
        st.dataframe(show, use_container_width=True, hide_index=True)

        if result["recommend_type"] == "용차" and best_own:
            st.caption(
                f"Best 자차: {best_own['vehicle']} · Objective Cost {best_own['objective_cost']:.3f} "
                f"> Fallback Cost {result['fallback_cost']:.2f}"
            )

with right:
    st.markdown("### AI 판단 근거")
    st.markdown(
        f'<div class="reason-box">{result["reason"]}</div>',
        unsafe_allow_html=True,
    )
    st.write("")
    st.markdown("**Hard Constraint**")
    st.write("✓ 15분 Buffer")
    st.write("✓ 다음 FIXED 일정 충돌 금지")
    st.write("✓ Stale State 120h")
    st.write(f"✓ Q95 이동범위 {engine.dist_q95:.1f}km / {engine.time_q95:.2f}h")
    st.write("")
    st.caption(
        "기본 Objective: 거리 0.60 · 시간 0.20 · 상차여유 0.10 · 복귀여유 0.10"
    )

st.divider()


# ============================================================
# Follow-up conversation
# ============================================================
st.markdown("### 💬 결과에 대해 추가로 요청해보세요")
st.caption(
    '예: "왜 이 차량이야?" · "2번 차량과 비교해줘" · '
    '"거리보다 시간을 더 중요하게 봐줘" · "다음 일정 여유를 우선해줘" · "이 차량으로 확정해줘"'
)

for m in st.session_state.messages[-10:]:
    if m.get("scope") == "followup":
        with st.chat_message(m["role"]):
            st.write(m["content"])

follow = st.chat_input("결과에 대해 질문하거나 조건을 변경하세요")
if follow:
    st.session_state.messages.append({"role": "user", "content": follow, "scope": "followup"})

    try:
        route = classify_followup(follow)
        intent = route.get("intent")

        if intent == "new_order":
            reset_for_new_order(keep_commits=True)
            st.rerun()

        elif intent == "reoptimize":
            if st.session_state.confirmed:
                answer = "이미 현재 배차를 확정했습니다. 상단의 '새 주문 입력' 버튼으로 다음 주문을 입력해주세요."
            else:
                preference = route.get("preference") or "balanced"
                if preference == "custom" and route.get("custom_weights"):
                    st.session_state.weights = route["custom_weights"]
                else:
                    st.session_state.weights = preset_weights(preference)
                rerun_current()
                w = st.session_state.weights
                answer = (
                    f"요청한 우선순위를 반영해 다시 계산했습니다. "
                    f"현재 가중치는 거리 {w['distance']:.2f}, 시간 {w['time']:.2f}, "
                    f"상차여유 {w['pickup_slack']:.2f}, 복귀여유 {w['return_slack']:.2f}입니다. "
                    f"새 추천은 {st.session_state.current_result['recommend_type']} "
                    f"{st.session_state.current_result.get('recommended_vehicle') or ''}입니다."
                )

        elif intent == "compare":
            rank = int(route.get("candidate_rank") or 2)
            cands = result.get("top_candidates", [])
            if len(cands) < rank:
                answer = f"현재 표시 가능한 {rank}번 후보가 없습니다."
            else:
                first = cands[0]
                other = cands[rank - 1]
                answer = (
                    f"1번 {first['vehicle']}은 공차 {first['empty_km']:.1f}km, "
                    f"{first['empty_h'] * 60:.0f}분, Objective Cost {first['objective_cost']:.3f}입니다. "
                    f"{rank}번 {other['vehicle']}은 공차 {other['empty_km']:.1f}km, "
                    f"{other['empty_h'] * 60:.0f}분, Objective Cost {other['objective_cost']:.3f}입니다. "
                    "현재 주문에서는 Objective Cost가 더 낮은 자차가 우선됩니다."
                )

        elif intent == "confirm":
            if not st.session_state.confirmed:
                commit = engine.build_commit(result)
                if not any(
                    c.get("order_id") == commit.get("order_id")
                    for c in st.session_state.commits
                ):
                    st.session_state.commits.append(commit)
                st.session_state.confirmed = True

            if result["recommend_type"] == "자차":
                answer = (
                    f"{result['recommended_vehicle']} 배차를 확정했습니다. "
                    "다음 주문에서는 이 확정 배차의 하차 위치와 예상 완료시각을 차량 State에 반영합니다."
                )
            else:
                answer = "용차로 확정했습니다. 다음 주문을 바로 입력할 수 있습니다."

        else:
            answer = explain_result(follow, result)

    except Exception as e:
        answer = f"요청 처리 중 오류가 발생했습니다: {e}"

    st.session_state.messages.append(
        {"role": "assistant", "content": answer, "scope": "followup"}
    )
    st.rerun()


if st.session_state.confirmed:
    st.success(
        "배차가 확정되었습니다. 상단의 '← 새 주문 입력'을 누르면 확정된 State를 유지한 채 다음 주문을 처리합니다."
    )


with st.expander("현재 주문 / 최적화 상세"):
    st.json(
        {
            "order": {k: str(v) for k, v in order.items()},
            "weights": result["weights"],
            "fallback_cost": result["fallback_cost"],
            "historical_candidate_n": result["historical_candidate_n"],
            "feasible_candidate_n": result["feasible_candidate_n"],
            "best_own_cost": result["best_own_cost"],
            "exclude_reasons": result["exclude_reasons"],
            "confirmed_dispatch_count": len(st.session_state.commits),
        }
    )


### `requirements.txt`

In [ ]:
%%writefile requirements.txt
streamlit>=1.37
pandas>=2.0
numpy>=1.24
openpyxl>=3.1
openai>=1.0


### `README.md`

In [ ]:
%%writefile README.md
# 배차 AI Demo - Final v3.0

Stateful Sequential Real-Time Dispatch Optimization 데모입니다.

## 구조

- `app.py`: Streamlit UI
- `optimizer.py`: Stateful Sequential 실시간 배차 엔진
- `ai_helper.py`: 자연어 주문 파싱 / 후속 대화
- `requirements.txt`: 실행 패키지

## 데이터 위치

기본값:

`./공모전\2026_해커톤`

필요 파일:

- `모델링_최종후보표.xlsx`
- `순차모델링_FIXED스케줄.xlsx`
- `순차모델링_OD캐시.xlsx`

## Final v3.0 기본값

- Objective Weight: 0.60 / 0.20 / 0.10 / 0.10
- Fallback Cost: 1.12
- Buffer: 15분
- Stale State cutoff: 120h
- Q75/Q95: OD cache의 양수 거리/시간 분포에서 자동 계산

## 실행

```bat
conda activate hackathon2026
cd /d "./공모전\2026_해커톤\dispatch_ai_demo"
python -m streamlit run app.py --server.address 127.0.0.1 --server.port 8501
```

브라우저:

`http://127.0.0.1:8501`

## 새 주문 버튼

결과 화면 상단의 `← 새 주문 입력`은 현재 결과 화면만 초기화합니다.
이미 `확정`한 배차 commit은 유지되므로 다음 주문의 차량 State에 반영됩니다.


### `VERSION.txt`

In [ ]:
%%writefile VERSION.txt
Dispatch AI Demo v3.0
Stateful Sequential Real-Time Dispatch
Fallback Cost = 1.12
NO SIDEBAR
NO 60-MIN HORIZON


## 5. 생성 파일 및 버전 확인

여기에서 **Fallback Cost 1.12 / NO SIDEBAR / Stateful Sequential** 문구가 확인되면 최종 버전이 맞습니다.

In [ ]:
from pathlib import Path

for fname in ["optimizer.py", "ai_helper.py", "app.py"]:
    source = Path(fname).read_text(encoding="utf-8")
    compile(source, fname, "exec")
    print(f"PASS: {fname}")

print("\n--- VERSION.txt ---")
print(Path("VERSION.txt").read_text(encoding="utf-8"))

print("--- 생성 위치 ---")
print(Path.cwd())


## 6. 배차 엔진 단독 로딩 테스트

Streamlit 화면이 흰 화면으로 보일 때는 먼저 이 셀로 데이터 엔진 자체가 정상적으로 로딩되는지 확인할 수 있습니다.

첫 실행은 Excel 파일을 읽기 때문에 시간이 걸릴 수 있습니다.

In [ ]:
import time
from optimizer import DispatchEngine

start = time.time()
print("엔진 로딩 시작...")

engine_test = DispatchEngine(
    base_dir=BASE_DIR,
    default_fallback_cost=1.12,
)

print(f"엔진 로딩 완료: {time.time() - start:.1f}초")
print(engine_test.data_summary())


## 7. Streamlit 실행

기존에 8501 포트에서 이전 Streamlit 앱이 실행 중이면 먼저 해당 터미널에서 `Ctrl + C`로 종료하는 것이 가장 안전합니다.

아래 셀은 **현재 `dispatch_ai_demo_v3_clean/app.py`**를 8501 포트에서 실행합니다.

In [ ]:
import subprocess
import sys
import os
import time

try:
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=5)
        print("이 노트북에서 실행한 이전 Streamlit 프로세스를 종료했습니다.")
except Exception:
    pass

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.address",
        "127.0.0.1",
        "--server.port",
        "8501",
    ],
    cwd=str(DEMO_DIR),
    env=os.environ.copy(),
)

time.sleep(2)

print("Streamlit 실행 요청 완료")
print("실행 파일:", DEMO_DIR / "app.py")
print("브라우저 주소: http://127.0.0.1:8501")
print("프로세스 PID:", streamlit_process.pid)


## 8. 데모에서 확인해야 할 최종 화면

정상적인 v3.0 화면이면 다음이 맞습니다.

### 없어야 하는 것
- 왼쪽 `운영 설정` Sidebar
- `Horizon 60분`
- `60분 Horizon 동시 고려`
- `Fallback Cost 1.08`

### 보여야 하는 것
- `Stateful Sequential` 실시간 배차
- Fallback Cost **1.12**
- 주문 1건 즉시 추천
- 결과 화면 상단 **`← 새 주문 입력`**
- 추천 차량 순위의 `Objective Cost`
- Hard Constraint 및 AI 판단 근거
- 확정한 차량 State를 유지한 채 다음 주문 입력 가능


## 9. 데모 테스트 문장

Historical Replay 데이터 범위에 맞춰 2026년 1~3월 시각을 사용하는 것을 권장합니다.

예시:

`2026-03-18 09:00 의왕ICD에서 부산신항으로 40HC 18톤 수출 건 배차해줘`

결과 후 예시:

- `왜 이 차량이야?`
- `2번 차량과 비교해줘`
- `거리보다 시간을 더 중요하게 봐줘`
- `다음 일정 여유가 많은 차량으로 다시 추천해줘`
- `이 차량으로 확정해줘`

배차 확정 후 결과 화면의 **`← 새 주문 입력`** 버튼을 눌러 다음 주문을 처리하면 됩니다.


## 10. Streamlit 종료

In [ ]:
try:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=5)
    print("Streamlit 프로세스를 종료했습니다.")
except NameError:
    print("현재 노트북에서 실행한 Streamlit 프로세스가 없습니다.")
except Exception as e:
    print("종료 중 메시지:", e)
